In [ ]:
import pandas as pd

apps_registry = [
    # iParitet
    {
        "bank": "Паритет",
        "app_name": "iParitet",
        "platform": "Google Play",
        "platform_app_id": "by.iparitet",
        "country": "by",
        "is_active": True
    },
    {
        "bank": "Паритет",
        "app_name": "iParitet",
        "platform": "App Store",
        "platform_app_id": "1543156711",
        "country": "by",
        "is_active": True
    },

    # INSNC
    {
        "bank": "Альфа-банк",
        "app_name": "INSNC: Мобильный банкинг, банк",
        "platform": "Google Play",
        "platform_app_id": "by.alfabank.app.insnc3",
        "country": "by",
        "is_active": True
    },
    {
        "bank": "Альфа-банк",
        "app_name": "INSNC",
        "platform": "App Store",
        "platform_app_id": "1633071765",
        "country": "by",
        "is_active": False
    },

    # Prior Online
    {
        "bank": "Приорбанк",
        "app_name": "Prior Online",
        "platform": "Google Play",
        "platform_app_id": "by.st.priormobile",
        "country": "by",
        "is_active": True
    },
    {
        "bank": "Приорбанк",
        "app_name": "Prior Online",
        "platform": "App Store",
        "platform_app_id": "521143601",
        "country": "by",
        "is_active": True
    },

    # МТБанк Moby
    {
        "bank": "МТБанк",
        "app_name": "МТБанк Moby",
        "platform": "Google Play",
        "platform_app_id": "by.mtbank.Moby",
        "country": "by",
        "is_active": True
    },
    {
        "bank": "МТБанк",
        "app_name": "MTbank Moby",
        "platform": "App Store",
        "platform_app_id": "1534853667",
        "country": "by",
        "is_active": True
    },

    # Белагропромбанк
    {
        "bank": "БАПБ",
        "app_name": "Belagroprombank",
        "platform": "Google Play",
        "platform_app_id": "com.alseda.bapb",
        "country": "by",
        "is_active": True
    },
    {
        "bank": "БАПБ",
        "app_name": "Belagroprombank",
        "platform": "App Store",
        "platform_app_id": "831489120",
        "country": "by",
        "is_active": False
    },

    # Белгазпромбанк
    {
    "bank": "Белгазпром",
    "app_name": "DailyFin",
    "platform": "Google Play",
    "platform_app_id": "by.bgpb.dailyfin",
    "country": "by",
    "is_active": True
    },
    {
    "bank": "Белгазпром",
    "app_name": "Zdraunik",
    "platform": "App Store",
    "platform_app_id": "6810668216",
    "country": "by",
    "is_active": False
    }
]

registry_df = pd.DataFrame(apps_registry)

# Формируем ссылки на страницы приложений
registry_df["store_url"] = registry_df.apply(
    lambda row: (
        f"https://play.google.com/store/apps/details?id={row['platform_app_id']}"
        if row["platform"] == "Google Play"
        else f"https://apps.apple.com/{row['country']}/app/id{row['platform_app_id']}"
    ),
    axis=1
)

display(registry_df)

,app_key,app_name,platform,platform_app_id,country,is_active,store_url
0,iparitet,iParitet,Google Play,by.iparitet,by,True,https://play.google.com/store/apps/details?id=...
1,iparitet,iParitet,App Store,1543156711,by,True,https://apps.apple.com/by/app/id1543156711
2,insnc,"INSNC: Мобильный банкинг, банк",Google Play,by.alfabank.app.insnc3,by,True,https://play.google.com/store/apps/details?id=...
3,insnc,INSNC,App Store,1633071765,by,False,https://apps.apple.com/by/app/id1633071765
4,prior_online,Prior Online,Google Play,by.st.priormobile,by,True,https://play.google.com/store/apps/details?id=...
5,prior_online,Prior Online,App Store,521143601,by,True,https://apps.apple.com/by/app/id521143601
6,mtbank_moby,МТБанк Moby,Google Play,by.mtbank.Moby,by,True,https://play.google.com/store/apps/details?id=...
7,mtbank_moby,MTbank Moby,App Store,1534853667,by,True,https://apps.apple.com/by/app/id1534853667
8,belagroprombank,Belagroprombank,Google Play,com.alseda.bapb,by,True,https://play.google.com/store/apps/details?id=...
9,belagroprombank,Belagroprombank,App Store,831489120,by,False,https://apps.apple.com/by/app/id831489120


In [ ]:
# Проверяем на дубли и кол-во записей в реестре
duplicates = registry_df.duplicated(
    subset=["platform", "platform_app_id"]
)

if duplicates.any():
    print("Внимание! Найдены повторяющиеся ID:")
    display(registry_df[duplicates])
else:
    print("Повторяющихся ID нет.")

print("Всего записей:", len(registry_df))
print("Приложений:", registry_df["bank"].nunique())
print("Активных платформ:", registry_df["is_active"].sum())

Повторяющихся ID нет.
Всего записей: 10
Приложений: 5
Активных платформ: 8


In [ ]:
# Функция-костыль на получение распределения оценок в App Store
from bs4 import BeautifulSoup
import requests
import json


def get_appstore_rating_distribution(app_id, country):
    """
    Получает распределение оценок App Store по конкретной стране.

    Возвращает:
        stars: [1 звезда, 2 звезды, 3 звезды, 4 звезды, 5 звёзд]
        total: общее количество оценок из того же источника
    """

    url = f"https://apps.apple.com/{country.lower()}/app/id{app_id}"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/131.0.0.0 Safari/537.36"
        )
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    script = soup.find(
        "script",
        id="serialized-server-data"
    )

    if script is None or not script.string:
        raise ValueError(
            "Не найден JSON с данными рейтинга App Store"
        )

    data = json.loads(script.string)

    # Рекурсивный поиск блока Ratings
    def find_ratings(obj):

        if isinstance(obj, dict):

            if (
                obj.get("$kind") == "Ratings"
                and str(obj.get("productId")) == str(app_id)
            ):
                return obj

            for value in obj.values():
                result = find_ratings(value)

                if result is not None:
                    return result

        elif isinstance(obj, list):

            for value in obj:
                result = find_ratings(value)

                if result is not None:
                    return result

        return None

    ratings_data = find_ratings(data)

    if ratings_data is None:
        raise ValueError(
            "Блок Ratings не найден в данных App Store"
        )

    raw_counts = ratings_data.get("ratingCounts")
    total = ratings_data.get("totalNumberOfRatings")
    average = ratings_data.get("ratingAverage")

    if not raw_counts or len(raw_counts) != 5:
        raise ValueError(
            "Некорректное распределение оценок"
        )

    if total is None:
        raise ValueError(
            "Не найдено общее количество оценок"
        )

    # Apple возвращает порядок: 5, 4, 3, 2, 1
    # Переводим в наш формат: 1, 2, 3, 4, 5

    stars = [
        int(round(x))
        for x in reversed(raw_counts)
    ]

    total = int(round(total))

    # Проверяем, что сумма распределения совпадает
    if abs(sum(stars) - total) > 2:
        raise ValueError(
            "Сумма распределения не совпадает с общим "
            "количеством оценок"
        )

    # Дополнительная проверка среднего рейтинга
    calculated_average = (
        sum(stars[i] * (i + 1) for i in range(5))
        / sum(stars)
    )

    if average is not None:
        if abs(calculated_average - float(average)) > 0.15:
            raise ValueError(
                "Средний рейтинг не соответствует распределению"
            )

    return stars, total

In [ ]:
import requests
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from google_play_scraper import app as google_play_app

# Время снимка — по Минску
snapshot_time = datetime.now(ZoneInfo("Europe/Minsk"))
snapshot_date_time = snapshot_time.strftime("%Y-%m-%d %H:%M:%S")
snapshot_day = snapshot_time.strftime("%Y-%m-%d")

# Берём только активные платформы
active_apps = registry_df[registry_df["is_active"] == True]

# Сюда будут записываться успешные результаты
collected_data = []

# Сюда — ошибки
collection_errors = []

# Сбор данных
for _, item in active_apps.iterrows():

    try:
        if item["platform"] == "Google Play":

            result_data = google_play_app(
                item["platform_app_id"],
                lang="ru",
                country=item["country"]
            )

            store_app_name = result_data.get("title")
            rating = result_data.get("score")
            total_ratings = result_data.get("ratings")
            version = result_data.get("version")

            histogram = result_data.get("histogram") or [None] * 5

            # Если распределение отсутствует или неполное
            histogram = list(histogram) + [None] * (5 - len(histogram))

            stars = histogram[:5]

        elif item["platform"] == "App Store":

            response = requests.get(
                "https://itunes.apple.com/lookup",
                params={
                    "id": item["platform_app_id"],
                    "country": item["country"]
                },
                timeout=30
            )

            response.raise_for_status()
            apple_data = response.json()

            if apple_data.get("resultCount", 0) == 0:
                raise ValueError("Приложение не найдено в App Store")

            result_data = apple_data["results"][0]

            store_app_name = result_data.get("trackName")
            rating = result_data.get("averageUserRating")
            version = result_data.get("version")
            total_ratings = result_data.get("userRatingCount")

            # Получаем распределение по стране
            stars, total_ratings_from_store = get_appstore_rating_distribution(
                item["platform_app_id"],
                item["country"]
            )

        else:
            raise ValueError("Неизвестная платформа")

        # Проверяем соответствие названия
        name_match = (
            str(item["app_name"]).strip().casefold()
            == str(store_app_name).strip().casefold()
        )

        # Формируем запись
        record = {
            "date_time": snapshot_date_time,
            "record_id": (
                f"{snapshot_day}_{item['bank']}_"
                f"{item['platform']}_{item['platform_app_id']}"
            ),
            "bank": item["bank"],
            "app": item["app_name"],
            "store_app_name": store_app_name,
            "platform": item["platform"],
            "platform_app_id": item["platform_app_id"],
            "store_url": item["store_url"],
            "country": item["country"],
            "rating": rating,
            "total_ratings": total_ratings,
            "stars_1": stars[0],
            "stars_2": stars[1],
            "stars_3": stars[2],
            "stars_4": stars[3],
            "stars_5": stars[4],
            "version": version,
            "name_match": name_match
        }

        collected_data.append(record)

        print(
            f"Успешно: {item['app_name']} — {item['platform']}"
        )

        if not name_match:
            print(
                f"  ВНИМАНИЕ: название в магазине "
                f"отличается: {store_app_name}"
            )

    except Exception as error:

        collection_errors.append({
            "bank": item["bank"],
            "app_name": item["app_name"],
            "platform": item["platform"],
            "platform_app_id": item["platform_app_id"],
            "error": str(error)
        })

        print(
            f"Ошибка: {item['app_name']} — "
            f"{item['platform']}: {error}"
        )


# Преобразуем результаты в DataFrame
collected_df = pd.DataFrame(collected_data)
errors_df = pd.DataFrame(collection_errors)

print("\nСбор завершён!")
print("Успешных записей:", len(collected_df))
print("Ошибок:", len(errors_df))

display(collected_df)

if not errors_df.empty:
    print("Ошибки сбора:")
    display(errors_df)

Успешно: iParitet — Google Play
Успешно: iParitet — App Store
Успешно: INSNC: Мобильный банкинг, банк — Google Play
Успешно: Prior Online — Google Play
Успешно: Prior Online — App Store
Успешно: МТБанк Moby — Google Play
Успешно: MTbank Moby — App Store
Успешно: Belagroprombank — Google Play

Сбор завершён!
Успешных записей: 8
Ошибок: 0


,date,record_id,app_key,app,store_app_name,platform,platform_app_id,store_url,country,rating,total_ratings,stars_1,stars_2,stars_3,stars_4,stars_5,version,name_match
0,2026-11-03 14:00:00,2026-11-03 14:00:00_iparitet_Google Play_by.ip...,iparitet,iParitet,iParitet,Google Play,by.iparitet,https://play.google.com/store/apps/details?id=...,by,4.906844,5654,53.0,20.0,31.0,181.0,5363.0,1.31,True
1,2026-11-03 14:00:00,2026-11-03 14:00:00_iparitet_App Store_1543156711,iparitet,iParitet,iParitet,App Store,1543156711,https://apps.apple.com/by/app/id1543156711,by,4.802130,3745,NaN,NaN,NaN,NaN,NaN,1.31,True
2,2026-11-03 14:00:00,2026-11-03 14:00:00_insnc_Google Play_by.alfab...,insnc,"INSNC: Мобильный банкинг, банк","INSNC: Мобильный банкинг, банк",Google Play,by.alfabank.app.insnc3,https://play.google.com/store/apps/details?id=...,by,3.627737,2783,750.0,151.0,110.0,131.0,1635.0,1.88.0,True
3,2026-11-03 14:00:00,2026-11-03 14:00:00_prior_online_Google Play_b...,prior_online,Prior Online,Prior Online,Google Play,by.st.priormobile,https://play.google.com/store/apps/details?id=...,by,4.731840,91516,3790.0,774.0,1189.0,4671.0,81087.0,3.49.4,True
4,2026-11-03 14:00:00,2026-11-03 14:00:00_prior_online_App Store_521...,prior_online,Prior Online,Prior Online,App Store,521143601,https://apps.apple.com/by/app/id521143601,by,4.826270,56667,NaN,NaN,NaN,NaN,NaN,3.49.3,True
5,2026-11-03 14:00:00,2026-11-03 14:00:00_mtbank_moby_Google Play_by...,mtbank_moby,МТБанк Moby,МТбанк Moby,Google Play,by.mtbank.Moby,https://play.google.com/store/apps/details?id=...,by,3.951456,17399,3630.0,485.0,601.0,1055.0,11623.0,1.0.200,True
6,2026-11-03 14:00:00,2026-11-03 14:00:00_mtbank_moby_App Store_1534...,mtbank_moby,MTbank Moby,MTbank Moby,App Store,1534853667,https://apps.apple.com/by/app/id1534853667,by,4.103460,3644,NaN,NaN,NaN,NaN,NaN,1.0.200,True
7,2026-11-03 14:00:00,2026-11-03 14:00:00_belagroprombank_Google Pla...,belagroprombank,Belagroprombank,Belagroprombank,Google Play,com.alseda.bapb,https://play.google.com/store/apps/details?id=...,by,4.680655,159261,7197.0,1606.0,3181.0,10883.0,136390.0,4.4.7,True


In [ ]:
# Проверка качества собранных данных

quality_report = collected_df[
    ["app", "platform", "rating", "total_ratings",
     "stars_1", "stars_2", "stars_3", "stars_4", "stars_5"]
].copy()

# Сумма распределения звёзд
quality_report["histogram_total"] = (
    quality_report[
        ["stars_1", "stars_2", "stars_3", "stars_4", "stars_5"]
    ].sum(axis=1, min_count=5)
)

# Разница между общим количеством оценок и распределением
quality_report["difference"] = (
    quality_report["total_ratings"]
    - quality_report["histogram_total"]
)

# Проверка полноты распределения
quality_report["histogram_available"] = (
    quality_report["histogram_total"].notna()
)

display(quality_report)

,app,platform,rating,total_ratings,stars_1,stars_2,stars_3,stars_4,stars_5,histogram_total,difference,histogram_available
0,iParitet,Google Play,4.906844,5654,53.0,20.0,31.0,181.0,5363.0,5648.0,6.0,True
1,iParitet,App Store,4.802130,3745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,"INSNC: Мобильный банкинг, банк",Google Play,3.627737,2783,750.0,151.0,110.0,131.0,1635.0,2777.0,6.0,True
3,Prior Online,Google Play,4.731840,91516,3790.0,774.0,1189.0,4671.0,81087.0,91511.0,5.0,True
4,Prior Online,App Store,4.826270,56667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
5,МТБанк Moby,Google Play,3.951456,17399,3630.0,485.0,601.0,1055.0,11623.0,17394.0,5.0,True
6,MTbank Moby,App Store,4.103460,3644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7,Belagroprombank,Google Play,4.680655,159261,7197.0,1606.0,3181.0,10883.0,136390.0,159257.0,4.0,True


In [ ]:
import os

# Создаём папку для проекта, если её ещё нет
history_path = "ratings_history.csv"

if os.path.exists(history_path):
    history_df = pd.read_csv(history_path)

    existing_ids = set(history_df["record_id"])

    new_rows = collected_df[
        ~collected_df["record_id"].isin(existing_ids)
    ].copy()

    history_df = pd.concat(
        [history_df, new_rows],
        ignore_index=True
    )

else:
    history_df = collected_df.copy()
    new_rows = collected_df.copy()

history_df.to_csv(history_path, index=False)

print("CSV успешно сохранён!")
print("Новых записей добавлено:", len(new_rows))
print("Всего записей в истории:", len(history_df))
print("Путь к файлу:", history_path)

CSV успешно сохранён!
Новых записей добавлено: 8
Всего записей в истории: 8
Путь к файлу: /content/drive/MyDrive/iParitet_monitoring/ratings_history_clean.csv


In [ ]:
def send_to_google_sheets(rows):
    if not rows:
        print("Новых записей для Google Sheets нет.")
        return

    response = requests.post(
        os.environ["GOOGLE_SCRIPT_URL"],
        json={
            "token": os.environ["API_TOKEN"],
            "rows": rows
        },
        timeout=60
    )

    response.raise_for_status()

    result = response.text.strip()

    if result != "OK":
        raise RuntimeError(
            f"Ошибка отправки в Google Sheets: {result}"
        )

    print("Google Sheets успешно обновлён.")


# Отправляем только новые записи, а не всю историю
rows_to_send = (
    new_rows
    .astype(object)
    .where(pd.notna(new_rows), None)
    .values
    .tolist()
)

send_to_google_sheets(rows_to_send)